# Phase 1 — Lesson 11: Singular Value Decomposition
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 11: Singular Value Decomposition
**Type:** Build | **Languages:** Python, Julia | **Time:** ~120 minutes
**Prerequisites:** Phase 1, Lessons 01, 02, 03



# What You'll Learn

---

## Learning Objectives

- Implement SVD via power iteration and explain the geometric meaning of U, Sigma, and V^T
- Apply truncated SVD for image compression and measure compression ratio vs reconstruction error
- Compute the Moore-Penrose pseudoinverse via SVD to solve overdetermined least-squares systems
- Connect SVD to PCA, recommendation systems, and Latent Semantic Analysis


# What SVD Does Geometrically

---

Every matrix, regardless of shape, performs three operations in sequence: rotate, scale, rotate. SVD makes this decomposition explicit.

```
A = U * Sigma * V^T

V^T rotates vectors in the input space (n-dimensional)
Sigma scales along each axis (stretches or compresses)
U rotates the result into the output space (m-dimensional)
```

SVD tells you: "This matrix takes a sphere of inputs, first rotates it by V^T, then stretches it into an ellipsoid by Sigma, then rotates the ellipsoid by U."


In [ ]:
import numpy as npimport matplotlib.pyplot as pltA = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)U, S, Vt = np.linalg.svd(A)# ReconstructSigma = np.zeros((3, 2)); np.fill_diagonal(Sigma, S)recon = U @ Sigma @ Vtprint(f"Max reconstruction error: {np.max(np.abs(A - recon)):.2e}")print(f"Singular values: {np.round(S, 4)}")

In [ ]:
import matplotlib.pyplot as plt# Low-rank approximations of a grayscale imagenp.random.seed(42)image = np.random.randn(100, 150)U, S, Vt = np.linalg.svd(image, full_matrices=False)fig, axes = plt.subplots(1, 4, figsize=(14, 3))for k, ax in zip([1, 5, 10, 50], axes):    Ak = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]    ax.imshow(Ak, cmap='gray', aspect='auto')    ax.set_title(f'Rank-{k}', fontweight='bold')    ax.set_xticks([]); ax.set_yticks([])plt.suptitle('SVD Low-Rank Approximations of an Image', fontsize=12, fontweight='bold')plt.tight_layout(); plt.show()for k in [1, 5, 10, 50]:    Ak = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]    err = np.linalg.norm(image - Ak) / np.linalg.norm(image)    print(f"k={k:>3d}  relative error={err:.4f}")

# The Outer Product Form

---

SVD can be written as a sum of rank-1 matrices:

```
A = sigma_1 * u_1 * v_1^T + sigma_2 * u_2 * v_2^T + ... + sigma_r * u_r * v_r^T
```

Each term adds one layer of structure. The first term captures the most important pattern. Truncating gives the best rank-k approximation (Eckart-Young theorem).


In [ ]:
import numpy as npA = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)U, S, Vt = np.linalg.svd(A)# Rank-1 approximationA1 = S[0] * np.outer(U[:, 0], Vt[0, :])print(f"Rank-1 approximation:\n{np.round(A1, 2)}")print(f"Original:\n{A}")

# SVD for Noise Reduction

---

Noisy data has signal concentrated in top singular values and noise spread across all. Truncating removes the noise floor.


In [ ]:
import numpy as npimport matplotlib.pyplot as pltnp.random.seed(42)clean = np.outer(np.sin(np.linspace(0, 4*np.pi, 100)), np.cos(np.linspace(0, 2*np.pi, 80)))noise = 0.3 * np.random.randn(100, 80)noisy = clean + noiseU, S, Vt = np.linalg.svd(noisy, full_matrices=False)denoised = U[:, :5] @ np.diag(S[:5]) @ Vt[:5, :]print(f"Noisy error:    {np.linalg.norm(noisy - clean):.4f}")print(f"Denoised error: {np.linalg.norm(denoised - clean):.4f}")print(f"Improvement:    {(1 - np.linalg.norm(denoised - clean) / np.linalg.norm(noisy - clean)):.1%}")

# Pseudoinverse via SVD

---

The Moore-Penrose pseudoinverse generalizes matrix inversion to non-square and singular matrices:

```
If A = U * Sigma * V^T, then:
A+ = V * Sigma+ * U^T
```

Solves least-squares problems: x = A+ b minimizes ||Ax - b||.


In [ ]:
import numpy as npA = np.array([[1, 1], [2, 1], [3, 1]], dtype=float)b = np.array([3, 5, 6], dtype=float)U, S, Vt = np.linalg.svd(A, full_matrices=False)S_inv = np.diag(1.0 / S)A_pinv = Vt.T @ S_inv @ U.Tx_svd = A_pinv @ bx_lstsq = np.linalg.lstsq(A, b, rcond=None)[0]print(f"SVD pseudoinverse:  {x_svd}")print(f"np.linalg.lstsq:    {x_lstsq}")

# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** Are singular values always non-negative?
> **Explanation:** Yes — and sorted in descending order.

**Q2:** What is the Eckart-Young theorem?
> **Explanation:** Truncated SVD is the provably optimal rank-k approximation.

**Q3:** What is SVD's connection to PCA?
> **Explanation:** PCA is SVD on centered data — principal components are right singular vectors.



# Practice Exercises

---

Do these in order:

**Exercise 1:** Compress a grayscale image using rank-k SVD for k = 1, 5, 10, 50; compare quality vs storage.

**Exercise 2:** Compute the pseudoinverse via SVD and solve an overdetermined system Ax = b.



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Singular value | How important this component is | Diagonal of Sigma — amount of stretch along each direction |
| Left singular vector | Output direction | Column of U — output-space basis vector |
| Right singular vector | Input direction | Column of V — input-space basis vector |
| Truncated SVD | Low-rank approximation | Keep only top k singular values — best rank-k approximation |
| Pseudoinverse | Generalized inverse | V Sigma+ U^T — solves least-squares for any matrix |
| Condition number | How trustworthy is this answer | sigma_max / sigma_min — sensitivity to input perturbations |


# Further Reading

---

- [3Blue1Brown: But what is the SVD?](https://www.youtube.com/watch?v=vSczTbgc8Rc)
- [Gilbert Strang: Linear Algebra and Its Applications, Chapter 7](https://math.mit.edu/~gs/linearalgebra/)


# Next Steps

---

When you're ready, continue to Lesson 12: Tensor Operations.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
